## Tool Calling<br>
Agent 将所有工具的描述与参数定义随请求发给model，model自行判断是否调用、调用哪个工具并输出调用指令，再由 Agent 解析指令执行对应工具并将结果回传给model循环处理直至得出最终回答。

![tool](../resources/img/tool.png)

In [1]:
# model
from rich import print as rprint
from dotenv import load_dotenv
load_dotenv(override=True)
from langchain.chat_models import init_chat_model
model = init_chat_model(
    model="deepseek-v4-flash", # 模型名称
)

In [14]:
# tool
from langchain_core.tools import tool

# Google 风格 Docstring
@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """
    获取城市的天气信息

    Args:
        city : 具体的城市

    Returns:
        返回城市的天气信息
    """
    return city + "晴天，温度15°C"


In [15]:
from langchain_core.utils.function_calling import convert_to_openai_tool

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取城市的天气信息',
        'parameters': {
            'properties': {'city': {'description': '具体的城市', 'type': 'string'}},
            'required': ['city'],
            'type': 'object'
        }
    }
}

### 1、用户提问今天天气如何

In [16]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# 用户问题
question = "苏州的天气如何？"
# Agent管理的消息队列
messages = [
    HumanMessage("苏州的天气如何？")
]

### 2、model决策是否调用工具 (分析用户问题)

In [32]:

# model绑定工具
model_with_tools = model.bind_tools([get_weather])

# model决策是否调用工具 (model不调用，只决策，agent来调用)
response = model_with_tools.invoke("苏州的天气如何？")
rprint(response.content)
rprint(response)

AIMessage(
    content='',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': '用户想知道苏州的天气情况。我可以调用get_weather工具来获取苏州的天气信息。'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 65,
            'prompt_tokens': 299,
            'total_tokens': 364,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 20,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256},
            'prompt_cache_hit_tokens': 256,
            'prompt_cache_miss_tokens': 43
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-v4-flash',
        'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
        'id': '632a489e-bc15-45d5-9736-529ce7bf83cd',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='lc_run--019f7371-8829-7242-9b2f-1aefda54320c-0',
    tool_calls=[
        {
            'name': 'get_weather',
            'args': {'city': '苏州'},
            'id': 'call_00_qtuPLFYuZOzs5YJpwaM29975',
            'type': 'tool_call'
        }
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 299,
        'output_tokens': 65,
        'total_tokens': 364,
        'input_token_details': {'cache_read': 256},
        'output_token_details': {'reasoning': 20}
    }
)

### 3、Agent收集model的决策 (AIMessage)

In [33]:
# Agent将model的AIMessage加入队列
messages.append(response)
rprint(messages)

[
    HumanMessage(content='苏州的天气如何？', additional_kwargs={}, response_metadata={}),
    AIMessage(
        content='',
        additional_kwargs={
            'refusal': None,
            'reasoning_content': '用户想知道苏州的天气情况。我可以调用get_weather工具来获取苏州的天气信息。'
        },
        response_metadata={
            'token_usage': {
                'completion_tokens': 65,
                'prompt_tokens': 299,
                'total_tokens': 364,
                'completion_tokens_details': {
                    'accepted_prediction_tokens': None,
                    'audio_tokens': None,
                    'reasoning_tokens': 20,
                    'rejected_prediction_tokens': None
                },
                'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256},
                'prompt_cache_hit_tokens': 256,
                'prompt_cache_miss_tokens': 43
            },
            'model_provider': 'deepseek',
            'model_name': 'deepseek-v4-flash',
            'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
            'id': '632a489e-bc15-45d5-9736-529ce7bf83cd',
            'finish_reason': 'tool_calls',
            'logprobs': None
        },
        id='lc_run--019f7371-8829-7242-9b2f-1aefda54320c-0',
        tool_calls=[
            {
                'name': 'get_weather',
                'args': {'city': '苏州'},
                'id': 'call_00_qtuPLFYuZOzs5YJpwaM29975',
                'type': 'tool_call'
            }
        ],
        invalid_tool_calls=[],
        usage_metadata={
            'input_tokens': 299,
            'output_tokens': 65,
            'total_tokens': 364,
            'input_token_details': {'cache_read': 256},
            'output_token_details': {'reasoning': 20}
        }
    )
]

### 4、Agent调用工具

In [34]:
# 模拟agent调用工具
for tool_call in response.tool_calls:
    if tool_call["name"] == "get_weather":
        tool_call_response = get_weather.invoke(tool_call)
        rprint(tool_call_response)
        rprint(type(tool_call_response))

ToolMessage(content='苏州晴天，温度15°C', name='get_weather', tool_call_id='call_00_qtuPLFYuZOzs5YJpwaM29975')

<class 'langchain_core.messages.tool.ToolMessage'>

### 5、将工具调用结果打包交给model

In [35]:
messages.append(tool_call_response) # 此时有3条message
rprint(messages)

[
    HumanMessage(content='苏州的天气如何？', additional_kwargs={}, response_metadata={}),
    AIMessage(
        content='',
        additional_kwargs={
            'refusal': None,
            'reasoning_content': '用户想知道苏州的天气情况。我可以调用get_weather工具来获取苏州的天气信息。'
        },
        response_metadata={
            'token_usage': {
                'completion_tokens': 65,
                'prompt_tokens': 299,
                'total_tokens': 364,
                'completion_tokens_details': {
                    'accepted_prediction_tokens': None,
                    'audio_tokens': None,
                    'reasoning_tokens': 20,
                    'rejected_prediction_tokens': None
                },
                'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256},
                'prompt_cache_hit_tokens': 256,
                'prompt_cache_miss_tokens': 43
            },
            'model_provider': 'deepseek',
            'model_name': 'deepseek-v4-flash',
            'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
            'id': '632a489e-bc15-45d5-9736-529ce7bf83cd',
            'finish_reason': 'tool_calls',
            'logprobs': None
        },
        id='lc_run--019f7371-8829-7242-9b2f-1aefda54320c-0',
        tool_calls=[
            {
                'name': 'get_weather',
                'args': {'city': '苏州'},
                'id': 'call_00_qtuPLFYuZOzs5YJpwaM29975',
                'type': 'tool_call'
            }
        ],
        invalid_tool_calls=[],
        usage_metadata={
            'input_tokens': 299,
            'output_tokens': 65,
            'total_tokens': 364,
            'input_token_details': {'cache_read': 256},
            'output_token_details': {'reasoning': 20}
        }
    ),
    ToolMessage(
        content='苏州晴天，温度15°C',
        name='get_weather',
        tool_call_id='call_00_qtuPLFYuZOzs5YJpwaM29975'
    )
]

In [38]:
final_response = model_with_tools.invoke(messages)
rprint(final_response)
rprint(final_response.content) # model最终的回复

AIMessage(
    content='苏州今天的天气情况如下：\n\n🌤 **天气：** 晴天  \n🌡 **温度：** 
15°C\n\n天气不错，是个晴朗的好天气！不过温度适中偏凉，出门的话建议带件外套哦～ 😊',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': '苏州的天气是晴天，温度15°C。我可以把这个信息告诉用户。'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 68,
            'prompt_tokens': 383,
            'total_tokens': 451,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 17,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256},
            'prompt_cache_hit_tokens': 256,
            'prompt_cache_miss_tokens': 127
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-v4-flash',
        'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
        'id': '33693631-685b-4d0e-8c75-b3cd1b851c42',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--019f7374-1669-7871-94f4-f77ad8d0605c-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 383,
        'output_tokens': 68,
        'total_tokens': 451,
        'input_token_details': {'cache_read': 256},
        'output_token_details': {'reasoning': 17}
    }
)

苏州今天的天气情况如下：

🌤 **天气：** 晴天  
🌡 **温度：** 15°C

天气不错，是个晴朗的好天气！不过温度适中偏凉，出门的话建议带件外套哦～ 😊

## 扩展